In [1]:
# %%
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("🚀 Step 1: Loading Processed Data...")

DATA_PROCESSED = os.path.join("..", "data", "processed")
file_path = os.path.join(DATA_PROCESSED, "processed_low_income_households.csv")

if not os.path.exists(file_path):
    raise FileNotFoundError(f"Missing {file_path}. Run the processing notebook first.")

df = pd.read_csv(file_path)
print(f"✅ Loaded {df.shape[0]} low-income households.")

/Users/zhenyuyue/miniforge3/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


🚀 Step 1: Loading Processed Data...
✅ Loaded 7450 low-income households.


In [2]:
# %%
print("✂️ Step 2: Splitting Data (70% Train, 15% Validate, 15% Test)...")

# Define features (X) and target (y)
# We drop SERIALNO (ID) and PUMA (too many categories for a simple linear model right now)
X = df.drop(columns=['SERIALNO', 'PUMA', 'y'])
y = df['y']

# 1st Split: Separate out 30% for our temporary holdout (Val + Test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# 2nd Split: Cut that 30% in half to get 15% Validate and 15% Test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"✅ Training set:   {X_train.shape[0]} rows")
print(f"✅ Validation set: {X_val.shape[0]} rows")
print(f"✅ Test set:       {X_test.shape[0]} rows")

✂️ Step 2: Splitting Data (70% Train, 15% Validate, 15% Test)...
✅ Training set:   5215 rows
✅ Validation set: 1117 rows
✅ Test set:       1118 rows


In [3]:
# %%
print("⚙️ Step 3: Scaling Features and Training Logistic Regression...")

# Logistic Regression needs features on the same scale (e.g., Income vs Number of Children)
scaler = StandardScaler()

# Fit the scaler ONLY on the training data to prevent data leakage, then transform all three
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Initialize and train the model
# class_weight='balanced' helps if the Gap/Claiming ratio is uneven
model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
model.fit(X_train_scaled, y_train)

print("✅ Model trained successfully!")

⚙️ Step 3: Scaling Features and Training Logistic Regression...
✅ Model trained successfully!


In [4]:
# %%
print("📊 Step 4: Evaluating on Validation Set...")

# Predict on validation data
y_val_pred = model.predict(X_val_scaled)

print("\n--- Validation Accuracy ---")
print(f"{accuracy_score(y_val, y_val_pred):.3f}")

print("\n--- Classification Report ---")
# 0 = In the Gap, 1 = Claiming SNAP
print(classification_report(y_val, y_val_pred, target_names=["The Gap (0)", "Claiming (1)"]))

print("\n--- Confusion Matrix ---")
print(confusion_matrix(y_val, y_val_pred))

📊 Step 4: Evaluating on Validation Set...

--- Validation Accuracy ---
0.711

--- Classification Report ---
              precision    recall  f1-score   support

 The Gap (0)       0.84      0.69      0.76       739
Claiming (1)       0.55      0.74      0.64       378

    accuracy                           0.71      1117
   macro avg       0.70      0.72      0.70      1117
weighted avg       0.74      0.71      0.72      1117


--- Confusion Matrix ---
[[513 226]
 [ 97 281]]


In [5]:
# %%
print("🔍 Step 5: What drives people into the GAP?")

# Extract coefficients and map them back to feature names
coefficients = model.coef_[0]
feature_names = X.columns

# Create a DataFrame for readability
df_coef = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
})

# Sort by impact
df_coef = df_coef.sort_values(by='Coefficient', ascending=False)

print("\nPositive numbers mean the feature INCREASES the chance of CLAIMING SNAP.")
print("Negative numbers mean the feature pushes people into THE GAP (Not claiming).\n")

display(df_coef)

🔍 Step 5: What drives people into the GAP?

Positive numbers mean the feature INCREASES the chance of CLAIMING SNAP.
Negative numbers mean the feature pushes people into THE GAP (Not claiming).



,Feature,Coefficient
14,NP,0.509667
2,HAS_DISABLED,0.405414
12,TEN,0.310316
3,NUM_CHILDREN,0.244854
6,IS_MINORITY_HH,0.170688
0,POVPIP,0.127876
10,GRPIP,0.004216
8,ACCESSINET,-0.037683
13,HINCP,-0.044968
1,HAS_ELDERLY,-0.100778


In [6]:
# %%
import statsmodels.api as sm
import pandas as pd

print("⚙️ Training Statistical Logistic Regression (with P-Values)...")

# 1. Convert scaled numpy arrays back to DataFrames to keep column names
X_train_df = pd.DataFrame(X_train_scaled, columns=X.columns)

# 2. Statsmodels requires us to explicitly add a constant (intercept)
X_train_sm = sm.add_constant(X_train_df)

# 3. Fit the Logit model
# We reset the index of y_train to ensure alignment with the new DataFrame
logit_model = sm.Logit(y_train.reset_index(drop=True), X_train_sm)

# Fit the model (disp=0 suppresses the raw optimization text)
result = logit_model.fit(disp=0) 

# 4. Create the comprehensive output table
df_coef_p = pd.DataFrame({
    'Feature': X_train_sm.columns,
    'Coefficient': result.params.values,
    'P-Value': result.pvalues.values
})

# Filter out the intercept, format, and sort by Coefficient
df_coef_p = df_coef_p[df_coef_p['Feature'] != 'const'].copy()
df_coef_p['Significant (p < 0.05)'] = df_coef_p['P-Value'] < 0.05

# Sort from most positive (Predicts Claiming) to most negative (Predicts Gap)
df_coef_p = df_coef_p.sort_values(by='Coefficient', ascending=False)

print("\n--- Feature Drivers ---")
display(df_coef_p)

# Uncomment the line below if you want the massive, traditional statistical summary block!
# print(result.summary())

⚙️ Training Statistical Logistic Regression (with P-Values)...

--- Feature Drivers ---


,Feature,Coefficient,P-Value,Significant (p < 0.05)
15,NP,0.479356,1.871259e-09,True
3,HAS_DISABLED,0.399790,2.205051e-32,True
13,TEN,0.324993,1.497885e-13,True
4,NUM_CHILDREN,0.241661,5.105775e-04,True
7,IS_MINORITY_HH,0.169520,9.764734e-07,True
1,POVPIP,0.112538,2.513195e-03,True
11,GRPIP,0.004871,9.052310e-01,False
14,HINCP,-0.043274,4.216649e-01,False
9,ACCESSINET,-0.044578,1.955476e-01,False
2,HAS_ELDERLY,-0.111066,7.398399e-03,True
